In [1]:
import sys
import torch

print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)
print("PyTorch location:", torch.__file__)

Python executable: C:\Users\prajin\Documents\kaggle competion\.venv\Scripts\python.exe
PyTorch version: 2.11.0+cu128
PyTorch location: C:\Users\prajin\Documents\kaggle competion\.venv\Lib\site-packages\torch\__init__.py


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
        "GB"
    )
else:
    print(torch.cuda.get_arch_list())

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.44 GB


In [3]:
from pathlib import Path
import pyarrow.parquet as pq

DATA_DIR = Path("data_set")

for filename in ["train.parquet", "test.parquet"]:
    path = DATA_DIR / filename

    print(f"\n--- {filename} ---")
    print("Exists:", path.exists())

    if path.exists():
        parquet_file = pq.ParquetFile(path)

        print("Rows:", parquet_file.metadata.num_rows)
        print("Row groups:", parquet_file.num_row_groups)
        print("Columns:", parquet_file.schema.names)
        print("Schema:")
        print(parquet_file.schema)


--- train.parquet ---
Exists: True
Rows: 2539608
Row groups: 21
Columns: ['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'element', 'element', 'num_peaks', 'base_peak_intensity', 'element', 'collision_energy_orig', 'collision_energy_orig_units']
Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 ingest_lib (String);
  optional binary field_id=-1 normalized_smiles (String);
  optional binary field_id=-1 inchikey (String);
  optional binary field_id=-1 inchikey14 (String);
  optional binary field_id=-1 molecular_formula (String);
  optional binary field_id=-1 ionization_mode (String);
  optional binary field_id=-1 instrument_type (String);
  optional binary field_id=-1 adduct (String);
  optional binary field_id=-1 adduct_orig (String);
  optional double field_id=-1 precursor_mz;
  optional double field_id=-1 precursor_error_ppm;

In [2]:
from pathlib import Path
from collections import Counter

import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path("data_set")
train_path = DATA_DIR / "train.parquet"
test_path = DATA_DIR / "test.parquet"

train_pf = pq.ParquetFile(train_path)
test_pf = pq.ParquetFile(test_path)

library_counts = Counter()
library_molecules = {}
unique_molecules = set()

for i in range(train_pf.num_row_groups):
    chunk = train_pf.read_row_group(
        i,
        columns=["ingest_lib", "inchikey14", "normalized_smiles"]
    ).to_pandas()

    library_counts.update(
        chunk["ingest_lib"].fillna("MISSING").value_counts().to_dict()
    )

    for library, group in chunk.groupby("ingest_lib", dropna=False):
        key = str(library)
        if key not in library_molecules:
            library_molecules[key] = set()
        library_molecules[key].update(
            group["inchikey14"].dropna().unique()
        )

    unique_molecules.update(chunk["inchikey14"].dropna().unique())

    print(
        f"Processed row group {i + 1}/{train_pf.num_row_groups}"
    )

library_summary = pd.DataFrame([
    {
        "library": lib,
        "spectra": count,
        "unique_molecules": len(library_molecules.get(lib, set()))
    }
    for lib, count in library_counts.items()
]).sort_values("spectra", ascending=False)

display(library_summary)

print("Total unique training molecules:", len(unique_molecules))

Processed row group 1/21
Processed row group 2/21
Processed row group 3/21
Processed row group 4/21
Processed row group 5/21
Processed row group 6/21
Processed row group 7/21
Processed row group 8/21
Processed row group 9/21
Processed row group 10/21
Processed row group 11/21
Processed row group 12/21
Processed row group 13/21
Processed row group 14/21
Processed row group 15/21
Processed row group 16/21
Processed row group 17/21
Processed row group 18/21
Processed row group 19/21
Processed row group 20/21
Processed row group 21/21


,library,spectra,unique_molecules
0,enveda-180,1153785,182941
7,pluskal_ms2,527581,46821
8,riken,347171,15892
2,gnps,220849,45750
3,massbank,101727,9180
5,mona,92416,11681
9,spectraverse,50933,9631
6,msdial,40765,9127
1,drug_plus,2545,2539
10,enveda-np-examples,1184,250


Total unique training molecules: 275810


In [4]:
test = pd.read_parquet(
    test_path,
    columns=[
        "molecule_id",
        "spectrum_id",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "instrument_type",
        "ms2_mzs",
        "ms2_normalized_intensities",
    ]
)

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Test spectra:", len(test))
print("Unique test molecules:", test["molecule_id"].nunique())
print("Unique spectrum IDs:", test["spectrum_id"].nunique())

print("\nSpectra per test molecule:")
display(test.groupby("molecule_id").size().describe())

print("\nSubmission columns:", sample_submission.columns.tolist())
display(sample_submission.head())

print("\nTest example:")
display(test.drop(columns=[
    "ms2_mzs", "ms2_normalized_intensities"
]).head())

Test spectra: 1213
Unique test molecules: 400
Unique spectrum IDs: 1213

Spectra per test molecule:


count    400.000000
mean       3.032500
std        1.382469
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        9.000000
dtype: float64


Submission columns: ['molecule_id', 'smiles']


,molecule_id,smiles
0,m_005e53,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
1,m_006153,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
2,m_00b5aa,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
3,m_01e922,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
4,m_0259d4,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...



Test example:


,molecule_id,spectrum_id,precursor_mz,adduct,ionization_mode,instrument_type
0,m_005e53,s_4d19d522,375.0709,[M+H]+,positive,timsTOF
1,m_005e53,s_5b6e28ff,375.0709,[M+H]+,positive,timsTOF
2,m_005e53,s_6e514292,375.0709,[M+H]+,positive,timsTOF
3,m_005e53,s_855ae19c,373.0562,[M-H]-,negative,timsTOF
4,m_006153,s_3c74e9fe,342.1569,[M+H]+,positive,timsTOF
